### Import

In [26]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [27]:
import os
import glob
from itertools import product
import numpy as np
import graphical_sampling as gs
import pandas as pd
import pickle
from tqdm import tqdm
from package_sampling.utils import inclusion_probabilities


In [28]:
os.chdir("/home/bardia/graphical-sampling")

### Reading Data

In [29]:
dfs ={}
path = "/home/bardia/graphical-sampling/simulations/populations"
files = ["aggregated.csv", "regular.csv", "meuse.csv", "swiss.csv"]

def popus(n):
    for f in files:
        full_path = os.path.join(path, f)
        df = pd.read_csv(full_path)
        N = len(df)
        
        if f in ["aggregated.csv", "regular.csv"]:
            coords = df.iloc[:, :].to_numpy()
            probs_ep  = np.repeat(n/N, N)
            name = f.replace('.csv', '')
            dfs[f'df_{name}'] = pd.DataFrame(
                {'coord_x':coords[:,0],
                'coord_y':coords[:,1],
                'pik_ep' : probs_ep,}
                )
        ##################################
        if f == "meuse.csv":
            coords = df[['x', 'y']].to_numpy()
            probs_up  = inclusion_probabilities(df['copper'].to_numpy(), n)
            probs_ep  = np.repeat(n/N, N)
            name = f.replace('.csv', '')
            dfs[f'df_{name}'] = pd.DataFrame(
                {'coord_x':coords[:,0],
                'coord_y':coords[:,1],
                'pik_ep' : probs_ep,
                'pik_up' : probs_up,
                'z'      : df['cadmium'], }
                )
        ###################################
        if f == "swiss.csv":
            coords = df[['x', 'y']].to_numpy()
            AREA = df['AREA'].to_numpy().clip(5,100)
            probs_up  = inclusion_probabilities(AREA, n)
            probs_ep  = np.repeat(n/N, N)
            name = f.replace('.csv', '')
            dfs[f'df_{name}'] = pd.DataFrame(
                {'coord_x':coords[:,0],
                'coord_y':coords[:,1],
                'pik_ep' : probs_ep,
                'pik_up' : probs_up,
                'z'      : df['AREA_A'].to_numpy(), }
                )
        ####################################
    return(dfs)


In [30]:
popu = 'df_swiss'
ep_or_up = 'pik_up'
n = 150
num_zones_cluster = [1]
num_zones_sweep = [(1, 1)]
num_splits = [4]


### Random Population and Parameters

In [31]:

df = popus(n)[popu]
N = len(df)

coords = pd.DataFrame(df[['coord_x', 'coord_y']])
inclusions = df[ep_or_up]
variable = df['z'] if 'z' in df.columns else inclusions

pop = gs.Population(coords, inclusions, n=n, variable=variable)
fbn = gs.clustering.FIPBalancedNMeans(n=n, init_clust_method = 'expanded', split_size = 0.001)
# fbn.fit(pop)
# fbn.fit_zones(num_zones=(1, 1), mode='sweep_xy')
# fbn.plot(mode='hard')
print(popu)

df_swiss


In [32]:
print(max(df['pik_up']))

0.8594126888272914


### Pre-Process

In [33]:
initial_designs = []
point_strategies = [
    #   'lexico_yx', 'angle', 'dist_from_origin',
    #  'dist_from_centroids', 'max_coord',
    # 'spiral', None,
    'lexico_xy', 'projection',
]

zone_strategies = [
    #   'lexico_yx', 'dist_from_origin',
    # 'spiral', None,
    'lexico_xy', 'projection'
]

zone_modes = [
    'cluster', 'sweep_xy', 'sweep_yx'
]


track = []

num_designs = len(point_strategies) * len(zone_strategies) * len(zone_modes) * len(num_splits)

for point_strategy, zone_strategy, zone_mode, num_split in tqdm(
    product(point_strategies, zone_strategies, zone_modes, num_splits), total=num_designs
):

    if zone_mode == 'cluster':
        list_of_num_zones = num_zones_cluster
    else:
        list_of_num_zones = num_zones_sweep

    for num_zones in list_of_num_zones:

        for _ in range(1):

            fbn = gs.clustering.FIPBalancedNMeans(n)
            fbn.fit(pop)
            # fbn.fit_zones(num_zones=num_zones, mode=zone_mode)

            # try:
            order = gs.Order.from_clusters(
                pop,
                fbn.clusters,
                point_strategy,
                zone_strategy,
                num_split
            )

            design = gs.Design.from_order(pop, order)

            initial_designs.append(design)

            track.append(
                (design.moran[0], design.moran[1], design, zone_strategy, point_strategy, zone_mode, num_zones)
            )
z = sorted(track, key=lambda x: x[0], reverse=False)
z[:1]

  0%|                                                                                            | 0/12 [00:00<?, ?it/s]/home/bardia/graphical-sampling/graphical_sampling/order.py:106: RuntimeWarning: Mean of empty slice.
  zone_centroids.append(coords.mean(axis=0))
/home/bardia/graphical-sampling/.venv/lib/python3.10/site-packages/numpy/_core/_methods.py:137: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
  8%|███████                                                                             | 1/12 [00:01<00:21,  1.94s/it]/home/bardia/graphical-sampling/graphical_sampling/order.py:106: RuntimeWarning: Mean of empty slice.
  zone_centroids.append(coords.mean(axis=0))
/home/bardia/graphical-sampling/.venv/lib/python3.10/site-packages/numpy/_core/_methods.py:137: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
 17%|██████████████                                                                      | 2/12 [00:06<00:32,  3.23s/it]/h

[(-0.15937861249844,
  0.022085503304423295,
  'projection',
  'lexico_xy',
  'cluster',
  1)]

In [34]:
initial_moran, initial_std, initial_design, zs, ps, zm, nz = z[0]


In [35]:
srs_var = N**2 * (1-n/N) * np.var(variable)/n

plain_flat = [
    N,
    n,
    5000,
    0,
    np.sum(variable),
    initial_design.nht_variance,
    srs_var / initial_design.nht_variance,
    0,
    initial_design.density_disparity[0],
    initial_design.density_disparity[1],
    initial_design.voronoi[0],
    initial_design.voronoi[1],
    initial_design.moran[0],
    initial_design.moran[1],
    initial_design.local_balance[0],
    initial_design.local_balance[1],
]

print(", ".join(map(str, plain_flat)))


959, 150, 5000, 0, 10255.32, 356792.43436261395, 16.816411258913433, 0, 0.004614867492269205, 0.028540123239664594, 0.1669128990952552, 0.020581102385760535, -0.15937861249844, 0.022085503304423295, 0.15429379096062984, 0.01778925246842176


In [36]:
criteria = gs.criteria.MoranCriteria()
gbfs = gs.search.GreedyBestFirstSearch(initial_designs, criteria)


In [37]:
import os
import pickle

save_dir_designs = "./simulations/best_design"
filename = f"initial_design_{popu}_{n}_{ep_or_up}.pkl"
filepath = os.path.join(save_dir_designs, filename)

mode = "save"   # "save" or "load"

if mode == "save":
    initial_design = gbfs.best_design
    with open(filepath, "wb") as f:
        pickle.dump(initial_design, f)

elif mode == "load":
    with open(filepath, "rb") as f:
        initial_design = pickle.load(f)

gbfs.best_designs = [initial_design]


### GRIZZLY

In [42]:

if 1==1:
    with open(filepath, "rb") as f:
        best_design = pickle.load(f)
        gbfs.initial_designs = [best_design]

In [43]:
gbfs.run(
    max_iterations=100,        # maximum number of GBFS search iterations (how long the algorithm explores)
    max_open_set_size=5,     # maximum number of candidate designs stored in the priority queue (search frontier)

    top_k=1,                    # number of best designs tracked during the search (best-so-far solutions)

    num_new_order_nodes=20,     # number of neighbors generated by modifying the ordering of samples
                                # (creates 20 candidate designs by changing sample order)

    num_new_exchange_nodes=0,   # number of neighbors generated using exchange moves
                                # (0 means no swapping of units between samples)

    num_clusters_range=(1,30),             # order changes are applied within up to 2 clusters
                                # (clusters are groups of zones processed together)

    num_zones_range=(1, 1),                # within a cluster, modify only 1 zone at a time

    num_changes_range=(1,3),              # each generated neighbor performs one order modification
                                # (e.g., one swap/reposition in the sequence)

    num_zone_changes=0,         # do not reassign samples between zones (zone structure remains fixed)

    pull_strategy='default',   # rule for selecting samples during exchange operations
                                # (not used here since exchange neighbors are disabled)

    exchange_coef=.75,          # intensity of unit swapping during exchanges
                                # (irrelevant here because num_new_exchange_nodes = 0)

    num_explore=1,               # number of designs expanded per iteration
                                # (1 means purely greedy: always expand the current best design)
    n_jobs = 1
)


--- Starting Parallel GBFS: Max Iterations=100, Batch Size=1, Workers=1 ---
Initial best criteria value: -0.1780
Iter     0/100 | Best: -0.1780 | Open:     1 | Closed:     0
0: -0.1780140,  order_change,  DSize:3387,  var:392268.31, Entp:6.41
1: -0.1780372,  order_change,  DSize:3387,  var:392308.74, Entp:6.41
1: -0.1781518,  order_change,  DSize:3387,  var:396774.78, Entp:6.41
1: -0.1784158,  order_change,  DSize:3387,  var:392174.19, Entp:6.41
2: -0.1786524,  order_change,  DSize:3387,  var:394013.11, Entp:6.41
3: -0.1788183,  order_change,  DSize:3387,  var:401264.58, Entp:6.41
3: -0.1794255,  order_change,  DSize:3387,  var:400899.14, Entp:6.41
4: -0.1796196,  order_change,  DSize:3387,  var:400821.25, Entp:6.41
5: -0.1796970,  order_change,  DSize:3387,  var:401423.09, Entp:6.41
5: -0.1797074,  order_change,  DSize:3387,  var:394950.73, Entp:6.41
5: -0.1801753,  order_change,  DSize:3387,  var:403338.48, Entp:6.41
6: -0.1802999,  order_change,  DSize:3387,  var:402931.56, Entp:6.4

KeyboardInterrupt: 

In [44]:

filename = f"best_design_{popu}_{n}_{ep_or_up}.pkl"
filepath = os.path.join(save_dir_designs, filename)

# save your design object
with open(filepath, "wb") as f:
    pickle.dump(gbfs.best_design, f)
gbfs.initial_designs = [gbfs.best_design]

In [ ]:
srs_var = N**2 * (1-n/N) * np.var(variable)/n
plain_flat = [
    N,
    n,
    5000,
    0,
    np.sum(variable),
    float(gbfs.best_design.nht_variance),
    float(srs_var / gbfs.best_design.nht_variance),
    0,
    *map(float, gbfs.best_design.density_disparity),
    *map(float, gbfs.best_design.voronoi),
    *map(float, gbfs.best_design.moran),
    *map(float, gbfs.best_design.local_balance),
]
arr = np.array(plain_flat, dtype=float)
simple = [float(x) for x in arr]
# Example: flatten all tuples in the list
flat = []
for x in simple:
    if isinstance(x, tuple):
        flat.extend(x)
    else:
        flat.append(x)

print(', '.join(str(x) for x in flat))




In [ ]:
sumi = 0
for i in gbfs.best_design.all_samples_and_probs[1]:
    sumi += i
    print(sumi)

# Store

In [ ]:
raise SystemExit("Stopping Run All: Archived cells below.")

In [ ]:
dir(gbfs.best_design)

In [ ]:

gbfs.best_design.entropy
len(gbfs.best_design.all_samples_and_probs[1])

gbfs.best_design.merge_identical()


print(len(gbfs.best_design.all_samples_and_probs[0]))

In [ ]:
gbfs.best_design